# Notebook 3: Chấm Điểm & Đánh Giá Chất Lượng 3 LoRA (Evaluation & Benchmark)
---
**Chuẩn bị:**
1. Ở cột Files bên trái Colab, tải lên file **`lora_models_3agents.zip`** (file 452MB bạn vừa tải về máy).
2. Tải lên 3 file test: `agent1_scope_test.jsonl`, `agent2_criteria_test.jsonl`, `agent3_pico_test.jsonl`.
3. Bấm **Run All**!

In [ ]:
# Cell 1: Tự động giải nén 3 bộ não LoRA (nếu có file zip)
import os
if os.path.exists("lora_models_3agents.zip"):
    print("📦 Đang tự động giải nén lora_models_3agents.zip...")
    !unzip -o lora_models_3agents.zip
    print("✅ Giải nén thành công 3 thư mục LoRA!")
else:
    print("ℹ️ Không thấy file zip, kiểm tra các thư mục LoRA...")


In [ ]:
# Cell 2: Cài đặt thư viện Unsloth
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes


In [ ]:
# Cell 3: Tải Base Model Llama-3-8B vào GPU
from unsloth import FastLanguageModel
import torch
import json
import time
from tqdm import tqdm

max_seq_length = 2048
dtype = None
load_in_4bit = True

print("⏳ Đang khởi tạo Llama 3 8B...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

prompt_template = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
"""
print("✅ Sẵn sàng chấm thi!")


In [ ]:
# Cell 4: Hàm chấm thi tự động và đo lường độ chính xác
def evaluate_agent(adapter_path, test_file, agent_name, required_keys):
    if not os.path.exists(adapter_path):
        print(f"❌ Không tìm thấy thư mục {adapter_path}! Hãy chắc chắn đã upload file zip.")
        return None
        
    print("\n=======================================================")
    print(f"📝 BẮT ĐẦU CHẤM THI: {agent_name.upper()}")
    print(f"📂 Đề thi: {test_file} | Bộ não: {adapter_path}")
    print("=======================================================")
    
    model_eval, _ = FastLanguageModel.from_pretrained(
        model_name = adapter_path,
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model_eval)
    
    with open(test_file, "r", encoding="utf-8") as f:
        test_samples = [json.loads(line) for line in f if line.strip()]
        
    total = len(test_samples)
    valid_json_count = 0
    valid_schema_count = 0
    total_time = 0
    sample_outputs = []
    
    for idx, item in enumerate(tqdm(test_samples, desc=f"Chấm {agent_name}")):
        prompt = prompt_template.format(item["instruction"], item["input"])
        inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")
        
        t0 = time.time()
        outputs = model_eval.generate(**inputs, max_new_tokens = 256, use_cache = True)
        dt = time.time() - t0
        total_time += dt
        
        full_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
        response_text = full_text.split("### Response:")[-1].strip()
        
        try:
            clean_json = response_text.replace("```json", "").replace("```", "").strip()
            data = json.loads(clean_json)
            valid_json_count += 1
            
            has_all_keys = any(k in data for k in required_keys)
            if has_all_keys:
                valid_schema_count += 1
                
            if idx < 3:
                sample_outputs.append({
                    "input": item["input"],
                    "ai_output": clean_json
                })
        except Exception:
            pass
            
    json_acc = (valid_json_count / total) * 100
    schema_acc = (valid_schema_count / total) * 100
    avg_speed = total_time / total
    
    print(f"\n📊 [KẾT QUẢ {agent_name.upper()}]:")
    print(f"- Tổng số câu thi:        {total} câu")
    print(f"- Chuẩn cú pháp JSON:     {json_acc:.1f}% ({valid_json_count}/{total})")
    print(f"- Đầy đủ trường Schema:   {schema_acc:.1f}% ({valid_schema_count}/{total})")
    print(f"- Tốc độ trung bình:      {avg_speed:.2f} giây/câu")
    
    return {
        "name": agent_name,
        "total": total,
        "json_acc": json_acc,
        "schema_acc": schema_acc,
        "avg_speed": avg_speed,
        "samples": sample_outputs
    }

res1 = evaluate_agent("lora_agent1_scope", "agent1_scope_test.jsonl", "Agent 1 (Scope)", ["status", "feedback", "suggested_topics", "scope"])
res2 = evaluate_agent("lora_agent2_criteria", "agent2_criteria_test.jsonl", "Agent 2 (Criteria)", ["include", "exclude"])
res3 = evaluate_agent("lora_agent3_pico", "agent3_pico_test.jsonl", "Agent 3 (Keywords & PICO)", ["P", "I", "boolean_query"])


In [ ]:
# Cell 5: In Báo Cáo Benchmark
report = f"""# 🏆 BÁO CÁO BENCHMARK 3 AGENTS (Llama-3-8B LoRA)
| Agent | Số câu thi | Chuẩn JSON (%) | Chuẩn Schema (%) | Tốc độ (giây/câu) |
| :--- | :---: | :---: | :---: | :---: |
| Agent 1 (Scope) | {res1["total"]} | {res1["json_acc"]:.1f}% | {res1["schema_acc"]:.1f}% | {res1["avg_speed"]:.2f}s |
| Agent 2 (Criteria) | {res2["total"]} | {res2["json_acc"]:.1f}% | {res2["schema_acc"]:.1f}% | {res2["avg_speed"]:.2f}s |
| Agent 3 (Keywords) | {res3["total"]} | {res3["json_acc"]:.1f}% | {res3["schema_acc"]:.1f}% | {res3["avg_speed"]:.2f}s |
"""
with open("BENCHMARK_SCORECARD.md", "w", encoding="utf-8") as f:
    f.write(report)
print(report)
print("✅ ĐÃ LƯU BÁO CÁO VÀO FILE: BENCHMARK_SCORECARD.md!")
